# Исследование процентных ставок России: RUONIA, ключевая ставка, КБД ОФЗ

Цель — проверить качество и поведение исторических рядов **до** калибровки моделей Васичека и CIR. Все ставки измеряются в **процентах годовых**, а разности ставок — в **процентных пунктах**. Графики строятся отдельно по календарям публикации; мы не заполняем выходные, праздники и отсутствующие ставки искусственными значениями.

**Запуск:** из корня репозитория установить пакет `python -m pip install -e ".[dev]"` и загрузить данные командой `python -m russian_rates --series all --start 2014-01-01`. Затем выполнить все ячейки ноутбука. Сохранённые графики и результаты намеренно очищены для удобства Git.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from russian_rates.analysis import (
    align_ruonia_with_key_rate,
    load_rates,
    period_statistics,
    quality_report,
)

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте Jupyter из папки репозитория russian-interest-rate-models.")
DATA_DIR = ROOT / "data" / "processed"

data = load_rates(DATA_DIR)
ruonia = data["ruonia"]
key_rate = data["key_rate"]
zcyc = data["zcyc"]
print(f"Источник: {DATA_DIR}")

## 1. Контроль качества исходных рядов

Количество строк **не равно** количеству дат для КБД: на каждую дату предусмотрены 12 сроков. Даты без публикаций могут быть нормальными (выходные, праздники, особенности торгов); максимальный календарный разрыв — лишь повод для проверки. Отсутствующий объём RUONIA допустим, отсутствующая ставка — нет.

In [ ]:
quality = quality_report(data)
display(quality)

if quality["duplicate_keys"].sum():
    raise ValueError("В исходных Parquet обнаружены дубликаты ключей: проверьте выгрузку.")
if quality["missing_rates"].sum() or quality["missing_dates"].sum():
    raise ValueError("Есть пропущенные ставки или даты: сначала проверьте выгрузку.")
if quality.loc[quality["series"].eq("zcyc"), "missing_zcyc_tenors"].sum():
    print("Внимание: для некоторых дат КБД доступно менее 12 сроков — проверьте источник.")

### Описательная статистика

Среднее за весь период и стандартное отклонение показывают свойства *выборки*, но ещё не являются долгосрочным равновесием или волатильностью модели Васичека.

In [ ]:
for series, column in (("ruonia", "ruonia_pct"), ("key_rate", "key_rate_pct"), ("zcyc", "yield_pct")):
    print(f"\n{series} ({column}, % годовых)")
    display(data[series][column].describe().to_frame(name=column).T)

## 2. История RUONIA и ключевой ставки

Построим графики по исходным наблюдениям, без общего `inner join` с КБД. В противном случае мы теряли бы часть наблюдений RUONIA.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ruonia["date"], ruonia["ruonia_pct"], label="RUONIA", linewidth=1.2)
ax.plot(key_rate["date"], key_rate["key_rate_pct"], label="Ключевая ставка", linewidth=1.2)
ax.set(title="RUONIA и ключевая ставка", ylabel="% годовых", xlabel="Дата")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 3. Отклонение RUONIA от ключевой ставки

Для каждой даты RUONIA используем последнюю **уже известную** ключевую ставку (`merge_asof(direction="backward")`). Не подставляем будущие значения. Если в начале истории ключевая ставка ещё не известна, отклонение будет `NaN`.

In [ ]:
aligned = align_ruonia_with_key_rate(ruonia, key_rate)
print(f"RUONIA наблюдений: {len(ruonia)}; после сопоставления: {len(aligned)}")
print(f"Дат без известной ключевой ставки на этот момент: {aligned['key_rate_pct'].isna().sum()}")
display(aligned[["spread_pp"]].describe().T)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(aligned["date"], aligned["spread_pp"], linewidth=0.9)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set(title="RUONIA − ключевая ставка", ylabel="Процентные пункты", xlabel="Дата")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 4. Кривая бескупонных доходностей ОФЗ

Для анализа формы кривой используем всю исходную историю КБД, а не только даты, совпавшие с RUONIA. `pivot` обнаружит дубликаты (одна дата и срок), вместо их незаметного усреднения.

In [ ]:
curve = zcyc.pivot(index="date", columns="tenor_years", values="yield_pct").sort_index()
groups = {
    "Короткие сроки": [0.25, 0.5, 0.75, 1.0],
    "Средние сроки": [2.0, 3.0, 5.0, 7.0],
    "Длинные сроки": [10.0, 15.0, 20.0, 30.0],
}
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
for ax, (title, tenors) in zip(axes, groups.items()):
    for tenor in tenors:
        if tenor in curve.columns:
            ax.plot(curve.index, curve[tenor], label=f"{tenor:g} г.", linewidth=0.9)
    ax.set(title=title, ylabel="% годовых")
    ax.grid(alpha=0.25)
    ax.legend(ncol=4)
axes[-1].set_xlabel("Дата")
fig.tight_layout()
plt.show()

## 5. Сравнение отдельных периодов

Выделим 2014–2015, 2020–2022 и **2023 — до последней даты данных**. Важно: нельзя ограничивать последний интервал датой `2026-01-01` — это исключит весь 2026 год. Экстремальные наблюдения 2014 и 2022 годов отражают реальные рыночные режимы: автоматически удалять их не следует.

In [ ]:
PERIODS = {
    "2014–2015": ("2014-01-01", "2016-01-01"),
    "2020–2022": ("2020-01-01", "2023-01-01"),
    "2023 — последняя дата": ("2023-01-01", None),
}
display(period_statistics(ruonia, PERIODS))

for label, (start, end) in PERIODS.items():
    start_ts = pd.Timestamp(start)
    ru = ruonia[ruonia["date"].ge(start_ts)]
    kr = key_rate[key_rate["date"].ge(start_ts)]
    crv = curve[curve.index >= start_ts]
    if end is not None:
        end_ts = pd.Timestamp(end)
        ru = ru[ru["date"].lt(end_ts)]
        kr = kr[kr["date"].lt(end_ts)]
        crv = crv[crv.index < end_ts]

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(ru["date"], ru["ruonia_pct"], label="RUONIA")
    axes[0].plot(kr["date"], kr["key_rate_pct"], label="Ключевая ставка")
    axes[0].set(ylabel="% годовых", title=f"Ставки: {label}")
    axes[0].legend()
    for tenor in [0.25, 1.0, 3.0, 10.0]:
        if tenor in crv.columns:
            axes[1].plot(crv.index, crv[tenor], label=f"{tenor:g} г.")
    axes[1].set(xlabel="Дата", ylabel="% годовых", title="КБД ОФЗ")
    axes[1].legend(ncol=4)
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

## 6. Неустойчивость среднего RUONIA

Скользящее среднее и стандартное отклонение по 252 **наблюдениям** (приблизительно один торговый год) позволяют увидеть смену рыночных режимов. Это не оценка `b` или `σ` модели Васичека: для них понадобится отдельная статистическая калибровка, в том числе с учётом шагов между датами.

In [ ]:
ru = ruonia.sort_values("date").set_index("date")["ruonia_pct"]
rolling_mean = ru.rolling(window=252, min_periods=126).mean()
rolling_std = ru.rolling(window=252, min_periods=126).std()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(ru.index, ru, label="RUONIA", alpha=0.55, linewidth=0.8)
axes[0].plot(rolling_mean.index, rolling_mean, label="Скользящее среднее (252)", linewidth=1.4)
axes[0].set(ylabel="% годовых", title="Изменение уровня ставок")
axes[0].legend()
axes[1].plot(rolling_std.index, rolling_std)
axes[1].set(ylabel="п.п.", xlabel="Дата", title="Скользящее стандартное отклонение (252)")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 7. Что делать дальше

1. Изучить периоды нестабильности и качество данных вручную; не удалять реальные процентные шоки как «ошибочные выбросы».
2. Подготовить частотные выборки RUONIA (например, по последнему доступному наблюдению каждой недели) и явно учитывать интервалы между датами.
3. Оценить Vasicek на реальной мере вероятности `P`; проверить чувствительность параметров `a`, `b` и `σ` к периоду калибровки.
4. Провести исторический тест прогнозов с переоценкой параметров во времени. Сравнить с прогнозом «ставка без изменений».

**Не следует** трактовать среднюю RUONIA за 2014–2026 годы как устойчивую фундаментальную «нормальную» ставку без проверки смены режимов.